# Commodity Price Forecasting Experiment

Rolling-origin evaluation of ARIMA and GARCH models against random walk baselines.

In [ ]:
import sys
sys.path.insert(0, '../src')

from models import *
from full_runner import FullRunner
from aggregator import ResultsAggregator

In [ ]:
# Configuration
COMMODITY = 'guitars'
DATA_PATH = '../data/interpolated_spiff_data.csv'
OUTPUT_ROOT = '../output'

HORIZON = 200
SLIDING_WINDOW_STEP = 100
START_FRACTION = 0.8
CI_LEVELS = [0.8, 0.9, 0.95]

In [ ]:
# Define model combinations
combos = [
    # Baselines
    (RWNoDrift, ConstantVar),
    (RWWithDrift, ConstantVar),
]

# ARIMA factory
def make_arima_ctor(p, q, trend):
    def ctor(transformer):
        return ARIMAForecaster(
            transformer,
            order_fn=lambda data: (p, 0, q),
            trend=trend
        )
    trend_label = 'TrendN' if trend == 'n' else 'TrendC'
    ctor.__name__ = f"ARIMA_p{p}_q{q}_{trend_label}"
    return ctor

# Add ARIMA models
for p in range(1, 4):
    for q in range(1, 4):
        for trend in ('n', 'c'):
            combos.append((make_arima_ctor(p, q, trend), ConstantVar))

In [ ]:
# Run evaluation
from tqdm.notebook import tqdm

for mean_ctor, var_ctor in tqdm(combos, desc="Model combos"):
    runner = FullRunner(
        commodity=COMMODITY,
        data_path=DATA_PATH,
        output_root=OUTPUT_ROOT,
        mean_ctor=mean_ctor,
        var_ctor=var_ctor,
        horizon=HORIZON,
        start_frac=START_FRACTION,
        step=SLIDING_WINDOW_STEP,
        ci_levels=CI_LEVELS
    )
    runner.run()

In [ ]:
# Aggregate results
agg_df = ResultsAggregator(
    root_dir=OUTPUT_ROOT,
    commodity=COMMODITY
).aggregate()

print(agg_df[['model', 'RMSE', 'MAE', 'MAPE']].head(10))